> 在线讲义：[第 19 章 · CuTe：用 (Shape, Stride) 描述一切数据布局](https://wikieden.github.io/cuda-to-tensor-core/lessons/19-cute-layout.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 19 章 · CuTe：用 (Shape, Stride) 描述一切数据布局

本章回答：第 13–18 章里到处是"第几个线程读哪个地址"的手算下标，CUTLASS 3/4 用什么统一地管理它们？CuTe 的 Layout 为什么只要一对 (Shape, Stride)？列优先、行优先、分块、按线程切分、swizzle 怎么都用同一套写法表达？

配套讲义：仓库里 `cuda-programming/notes/lessons/19-cute-layout.md`（讲解、动画、练习都在讲义里）。对应原文：CuTe 文档（`materials/code/cutlass/cute-docs/`）：`01_layout.md`（569 行，构造第 130–220 行、坐标与 colexicographical 顺序第 372–400 行）、`02_layout_algebra.md`（composition 第 77 行、divide 第 381 行、product 第 482 行）、`03_tensor.md`（partition 第 276–333 行）、`0x_gemm_tutorial.md`（sgemm_1 → sgemm_sm80）；GPU MODE L57 CuTe（`text/gpu-mode-lecture-57-cute.txt`）；Colfax WGMMA / TMA 博客里的 CuTe 用法

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 三种 Stride | index = $\sum$ 坐标 × 步长 | `Layout((4, 2), (1, 4))(c)` 等 | (1,0)→1/2/1，(0,1)→4/1/5，(3,1)→7/7/8 |
| 第一步 默认 Stride | LayoutLeft | `Layout((4, 2))` | (4, 2):(1, 4) |
| 第二步 colexicographic | 最左边的维变化最快 | `idx2crd(i, (4, 2))` | 0..7 → (0,0)(1,0)(2,0)(3,0)(0,1)… |
| 第二步 嵌套 | `(4,(2,2)):(2,(1,8))` | `Layout(...)((1, (1, 1)))` | 2 + 1 + 8 = 11 |
| 第三步 按 tile 切块 | `zipped_divide` | `zipped_divide_2d(8, 8, (4, 4))` | 形状 ((4,4),(2,2))；块 (1,0) 等于直接切片 |
| 第五步 swizzle | `Swizzle<3,3,3>` | `swizzle(i)` | 0..511 的一个排列 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `_flatten`**（来自 `ch19_cute_layout.py`）：把嵌套的 tuple 摊平成一串整数。

In [ ]:
def _flatten(t):
    """把嵌套的 tuple 摊平成一串整数。"""
    if isinstance(t, int):
        return [t]
    return [x for e in t for x in _flatten(e)]

**函数 `idx2crd`**（来自 `ch19_cute_layout.py`）：colexicographic 顺序：最左边的维变化最快。对嵌套 Shape 递归拆。

In [ ]:
def idx2crd(i: int, shape):
    """colexicographic 顺序：最左边的维变化最快。对嵌套 Shape 递归拆。"""
    if isinstance(shape, int):
        return i
    out = []
    for s in shape:
        n = int(np.prod(_flatten(s)))
        out.append(idx2crd(i % n, s))
        i //= n
    return tuple(out)

**函数 `left_strides`**（来自 `ch19_cute_layout.py`）：不写 Stride 时的默认 LayoutLeft：步长是 Shape 从左往右的累乘（广义列优先）。

In [ ]:
def left_strides(shape) -> tuple:
    """不写 Stride 时的默认 LayoutLeft：步长是 Shape 从左往右的累乘（广义列优先）。"""
    out, acc = [], 1
    for s in shape:
        out.append(acc)
        acc *= s
    return tuple(out)

**类 `Layout`**（来自 `ch19_cute_layout.py`）：Layout = (Shape, Stride)：坐标 → 下标 = 坐标逐项乘步长再相加。Shape / Stride 可以嵌套。

In [ ]:
class Layout:
    """Layout = (Shape, Stride)：坐标 → 下标 = 坐标逐项乘步长再相加。Shape / Stride 可以嵌套。"""

    def __init__(self, shape, stride=None):
        self.shape = shape
        self.stride = stride if stride is not None else left_strides(shape)

    def size(self) -> int:
        return int(np.prod(_flatten(self.shape)))

    def __call__(self, coord):
        """coord 可以是和 Shape 同样嵌套的坐标，也可以是一个一维整数（按 colexicographic 顺序换成多维坐标）。"""
        if isinstance(coord, int):
            coord = idx2crd(coord, self.shape)
        return sum(c * d for c, d in zip(_flatten(coord), _flatten(self.stride)))

    def cosize(self) -> int:
        """用到的最大下标 + 1（存储要开多大）。"""
        return max(self(i) for i in range(self.size())) + 1

    def __repr__(self):
        return f"{self.shape}:{self.stride}"

**函数 `swizzle`**（来自 `ch19_cute_layout.py`）：CuTe 的 Swizzle<B, M, S>：取下标第 [M+S, M+S+B) 位，异或到第 [M, M+B) 位上。

In [ ]:
def swizzle(x: int, b: int = 3, m: int = 3, s: int = 3) -> int:
    """CuTe 的 Swizzle<B, M, S>：取下标第 [M+S, M+S+B) 位，异或到第 [M, M+B) 位上。"""
    mask = ((1 << b) - 1) << (m + s)
    return x ^ ((x & mask) >> s)

**函数 `zipped_divide_2d`**（来自 `ch19_cute_layout.py`）：8 × 8 列优先矩阵按 4 × 4 分块：结果 [块内坐标 (i, j), 块号 (bx, by)] → 原下标，形状 ((4, 4), (2, 2))。

In [ ]:
def zipped_divide_2d(rows: int, cols: int, tile: tuple[int, int]) -> np.ndarray:
    """8 × 8 列优先矩阵按 4 × 4 分块：结果 [块内坐标 (i, j), 块号 (bx, by)] → 原下标，形状 ((4, 4), (2, 2))。"""
    L = Layout((rows, cols))                      # (8, 8):(1, 8)
    th, tw = tile
    out = np.zeros((th, tw, rows // th, cols // tw), dtype=int)
    for bx in range(rows // th):
        for by in range(cols // tw):
            for i in range(th):
                for j in range(tw):
                    out[i, j, bx, by] = L((bx * th + i, by * tw + j))
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 同一个 4 × 2，三种 Stride")
coords = [(0, 0), (1, 0), (0, 1), (3, 1)]
for L in (Layout((4, 2), (1, 4)), Layout((4, 2), (2, 1)), Layout((4, 2), (1, 5))):
    print(f"  {L!s:<14}", {c: L(c) for c in coords})
print("  不写 Stride：", Layout((4, 2)), "（默认左边快，和 C 数组的行优先相反）")

In [ ]:
print("\n第二步 colexicographic：Shape (4, 2) 的一维坐标 0..7")
print("  ", [idx2crd(i, (4, 2)) for i in range(8)])
print("  嵌套 (4, (2, 2)):(2, (1, 8))：坐标 (1, (1, 1)) →", Layout((4, (2, 2)), (2, (1, 8)))((1, (1, 1))))

In [ ]:
print("\n第三步 8 × 8 按 4 × 4 分块：块 (1, 0) 的块内下标")
d = zipped_divide_2d(8, 8, (4, 4))
print("  结果形状 ((4, 4), (2, 2)) =", d.shape)
print(d[:, :, 1, 0])
full = np.arange(64).reshape(8, 8, order="F")
print("  和直接切片 full[4:8, 0:4] 一样：", np.array_equal(d[:, :, 1, 0], full[4:8, 0:4]))

In [ ]:
print("\n第五步 Swizzle<3, 3, 3>：0..511 映射后仍是 0..511 的一个排列")
sw = [swizzle(i) for i in range(512)]
print("  前 10 个：", sw[:10], "；第 64..73 个：", sw[64:74])
print("  一一对应：", sorted(sw) == list(range(512)))

## 3. 代码里才看得到的

1. **NumPy 默认是行优先，CuTe 默认是列优先**：验证分块时要写 `np.arange(64).reshape(8, 8, order="F")`（F = Fortran 顺序，即列优先），不然对不上。
2. **嵌套坐标先摊平再点乘**：`(1, (1, 1))` 摊平成 1、1、1，和摊平的步长 2、1、8 点乘——这就是"层次化 Shape"在计算上的全部意思。
3. **`zipped_divide_2d` 只是 2D 的特例**：CuTe 的 `logical_divide` 对任意嵌套 layout 都成立，靠的是 complement 和 composition；这里只演示结果长什么样。

## 4. 自己改一改

- 把 `zipped_divide_2d(8, 8, (4, 4))` 换成按 2 × 4 分块，看结果的形状和块 (1, 1) 的下标。
- 把 `swizzle` 的参数换成 (2, 3, 3)、(3, 4, 3)，看前 80 个数怎么排。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 下标都是坐标点乘步长：Layout = Shape : Stride；
> 不写步长默认左边快（列优先），和 C 数组反着来；
> 形状能嵌套，一维也能拆两半；
> 分块是除法，铺满是乘法，重排是复合，swizzle 也是一个 layout；
> Tensor = 指针 + Layout；硬件指令包成 Atom，换卡只换 Atom。